# load_openaire_researchproduct_organizations

Prototipo del nodo `load_openaire_researchproduct_organizations` del pipeline `load_openaire`. No guarda datasets.


In [ ]:
from datetime import date
import pandas as pd

%load_ext kedro.ipython


In [ ]:
df_researchproduct_raw = catalog.load('raw/openaire/researchproduct/parquet/researchproduct_dev')
df_researchproduct_raw.head(2)


In [ ]:
def _add_openaire_extracted_metadata(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in _EXTRACTED_META_COLS:
        if col not in df.columns:
            df[col] = pd.NA
    return df


In [ ]:
def _add_openaire_loaded_metadata(df: pd.DataFrame, load_datetime=None) -> pd.DataFrame:
    df = df.copy()
    if load_datetime is None:
        load_datetime = date.today()
    df["_load_datetime"] = load_datetime
    return df


In [ ]:
def load_openaire_researchproduct_organizations(df: pd.DataFrame)-> pd.DataFrame:
    df = _add_openaire_extracted_metadata(df)

    df_research_organization = df[['id', 'organizations', *_EXTRACTED_META_COLS]].explode('organizations').reset_index(drop=True)
    df_research_organization.rename(columns={'id':'researchproduct_id'}, inplace=True)

    df_organizations = pd.json_normalize(df_research_organization['organizations'])
    df_organizations.rename(columns={'id':'organization_id'}, inplace=True)

    df_research_organization = pd.concat(
        [
            df_research_organization[['researchproduct_id', *_EXTRACTED_META_COLS]].reset_index(drop=True),
            df_organizations['organization_id'].reset_index(drop=True),
        ],
        axis=1
    )

    df_organization_pid = df_organizations.loc[:, ['organization_id', 'pids']].copy()
    df_organizations.drop(columns=['pids'], inplace=True)
    df_organization_pid.dropna(inplace=True)

    df_organization_pid = df_organization_pid.explode('pids', ignore_index=True)
    df_organization_pid.loc[:, ['organization_id', 'pids']]

    df_pid = pd.json_normalize(df_organization_pid['pids'])
    df_pid.rename(columns={'scheme':'pid_scheme','value':'pid_value'}, inplace=True)

    df_organization_pid.drop(columns=['pids'], inplace=True)
    df_organization_pid = pd.concat([df_organization_pid, df_pid], axis=1)

    df_organizations = (
        df_organizations
        .drop_duplicates(subset="organization_id", keep="first")
        .reset_index(drop=True)
    )
    
    meta_vals = {
        col: (df_research_organization[col].iloc[0] if len(df_research_organization) else pd.NA)
        for col in _EXTRACTED_META_COLS
    }
    for col, val in meta_vals.items():
        df_organizations[col] = val
        df_organization_pid[col] = val

    df_organizations = _add_openaire_loaded_metadata(df_organizations)
    df_research_organization = _add_openaire_loaded_metadata(df_research_organization)
    df_organization_pid = _add_openaire_loaded_metadata(df_organization_pid)

    return df_organizations, df_research_organization, df_organization_pid


In [ ]:
df_organizations, df_research_organization, df_organization_pid = load_openaire_researchproduct_organizations(df_researchproduct_raw)


In [ ]:
pd.DataFrame([
    {'dataset': 'df_organizations', 'rows': len(df_organizations), 'columns': len(df_organizations.columns)},
    {'dataset': 'df_research_organization', 'rows': len(df_research_organization), 'columns': len(df_research_organization.columns)},
    {'dataset': 'df_organization_pid', 'rows': len(df_organization_pid), 'columns': len(df_organization_pid.columns)},
])


In [ ]:
df_organizations.head(2)
